# Phase 3 - SQLite + SQL Queries

Stand up `data/chicago.db` with three tables (`weather`, `crime_daily`, `features`), index them on `date`, then run six SQL queries that show off JOINs, CASE WHEN, window functions, and a CTE.

In [1]:
import sys
sys.path.append("../src")
import sqlite3
import pandas as pd
from features import build_features

merged = pd.read_csv("../data/processed/merged.csv", parse_dates=["date"])
print("merged:", merged.shape)

merged: (2557, 18)


Split the merged dataset back into the two natural sub-tables (`weather` and `crime_daily`), and build the `features` table on top of merged.

In [2]:
weather_cols = ["date", "tmax_c", "tmin_c", "tmax_f", "tmin_f",
                "prcp_mm", "snow_mm", "snwd_mm", "awnd_ms"]
crime_cols = ["date", "total_crimes", "theft", "battery", "burglary",
              "assault", "narcotics", "criminal_damage",
              "arrest_count", "domestic_count"]

weather = merged[weather_cols].copy()
crime_daily = merged[crime_cols].copy()

feat = build_features(merged)
feat_cols = ["date", "tmax_f", "tmin_f", "prcp_mm", "awnd_ms",
             "day_of_week", "month", "year", "is_weekend", "is_holiday",
             "temp_range_f", "is_rainy", "is_hot", "is_cold",
             "tmax_f_lag1", "prcp_mm_lag1", "crime_7day_avg",
             "is_covid_year", "is_nye", "is_july4", "total_crimes"]
features = feat[feat_cols].copy()

# store dates as YYYY-MM-DD text so sqlite's strftime() works
for d in (weather, crime_daily, features):
    d["date"] = pd.to_datetime(d["date"]).dt.strftime("%Y-%m-%d")

print("weather:", weather.shape, "| crime_daily:", crime_daily.shape, "| features:", features.shape)

weather: (2557, 9) | crime_daily: (2557, 10) | features: (2557, 21)


## Build the database

Drop existing tables (so re-running the notebook is idempotent), then write each dataframe with `to_sql`. Add a date index to each table - speeds up the JOINs and ORDER BY in the queries below.

In [3]:
import os
os.makedirs("../data", exist_ok=True)
conn = sqlite3.connect("../data/chicago.db")
cur = conn.cursor()

for t in ["weather", "crime_daily", "features"]:
    cur.execute(f"DROP TABLE IF EXISTS {t}")

weather.to_sql("weather", conn, index=False)
crime_daily.to_sql("crime_daily", conn, index=False)
features.to_sql("features", conn, index=False)

cur.execute("CREATE INDEX idx_weather_date ON weather(date)")
cur.execute("CREATE INDEX idx_crime_date ON crime_daily(date)")
cur.execute("CREATE INDEX idx_features_date ON features(date)")
conn.commit()

for t in ["weather", "crime_daily", "features"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn).iloc[0, 0]
    print(f"{t}: {n} rows")

weather: 2557 rows
crime_daily: 2557 rows
features: 2557 rows


## Six SQL queries

Each one is meant to demo a different SQL technique.

### Q1 - Average crime by month

Simple GROUP BY with date extraction.

In [4]:
q1 = """
SELECT strftime('%m', date) AS month,
       ROUND(AVG(total_crimes), 1) AS avg_crime,
       COUNT(*) AS n_days
FROM crime_daily
GROUP BY month
ORDER BY month
"""
pd.read_sql(q1, conn)

,month,avg_crime,n_days
0,01,615.8,217
1,02,610.9,198
2,03,621.8,217
3,04,620.4,210
4,05,689.0,217
5,06,721.0,210
6,07,736.1,217
7,08,729.1,217
8,09,719.1,210
9,10,697.8,217


### Q2 - Top 10 hottest days and their crime counts (JOIN)

Joins `weather` to `crime_daily` on `date`.

In [5]:
q2 = """
SELECT w.date, w.tmax_f, c.total_crimes, c.theft, c.battery
FROM weather w
JOIN crime_daily c ON w.date = c.date
ORDER BY w.tmax_f DESC
LIMIT 10
"""
pd.read_sql(q2, conn)

,date,tmax_f,total_crimes,theft,battery
0,2023-08-24,100.04,776,179,142
1,2022-06-21,98.96,657,170,105
2,2024-08-27,98.96,713,175,136
3,2022-06-14,98.06,665,163,124
4,2023-08-23,98.06,705,152,113
5,2018-05-27,96.98,884,178,247
6,2018-08-04,96.98,907,244,201
7,2020-08-24,96.98,637,139,131
8,2024-06-17,96.98,841,182,191
9,2018-06-29,96.08,782,195,148


### Q3 - Average crime on rainy vs. dry days (CASE WHEN)

Rainy = > 1 mm precipitation, matching the `is_rainy` flag I use later.

In [6]:
q3 = """
SELECT
    CASE WHEN w.prcp_mm > 1 THEN 'rainy' ELSE 'dry' END AS rain_status,
    ROUND(AVG(c.total_crimes), 1) AS avg_crime,
    COUNT(*) AS n_days
FROM weather w
JOIN crime_daily c ON w.date = c.date
GROUP BY rain_status
"""
pd.read_sql(q3, conn)

,rain_status,avg_crime,n_days
0,dry,675.5,1890
1,rainy,658.4,667


### Q4 - Per-category totals by season (CASE WHEN + GROUP BY)

In [7]:
q4 = """
SELECT
    CASE
        WHEN CAST(strftime('%m', date) AS INTEGER) IN (12, 1, 2) THEN 'winter'
        WHEN CAST(strftime('%m', date) AS INTEGER) IN (3, 4, 5)  THEN 'spring'
        WHEN CAST(strftime('%m', date) AS INTEGER) IN (6, 7, 8)  THEN 'summer'
        ELSE 'fall'
    END AS season,
    SUM(theft) AS theft,
    SUM(battery) AS battery,
    SUM(burglary) AS burglary,
    SUM(assault) AS assault,
    SUM(narcotics) AS narcotics,
    SUM(criminal_damage) AS criminal_damage
FROM crime_daily
GROUP BY season
ORDER BY CASE season
    WHEN 'winter' THEN 1 WHEN 'spring' THEN 2
    WHEN 'summer' THEN 3 ELSE 4 END
"""
pd.read_sql(q4, conn)

,season,theft,battery,burglary,assault,narcotics,criminal_damage
0,winter,86978,68180,13889,31548,15419,40504
1,spring,88256,78752,14117,36596,14806,47036
2,summer,107619,87884,16347,41045,14088,53847
3,fall,100071,77921,15969,37378,13474,48994


### Q5 - 7-day rolling average via window function

Just showing the first month of 2018 so the output is readable.

In [8]:
q5 = """
SELECT date, total_crimes,
       ROUND(AVG(total_crimes) OVER (
           ORDER BY date
           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
       ), 1) AS rolling_7d
FROM crime_daily
WHERE date BETWEEN '2018-01-01' AND '2018-01-31'
ORDER BY date
"""
pd.read_sql(q5, conn)

,date,total_crimes,rolling_7d
0,2018-01-01,1038,1038.0
1,2018-01-02,564,801.0
2,2018-01-03,576,726.0
3,2018-01-04,601,694.8
4,2018-01-05,666,689.0
5,2018-01-06,589,672.3
6,2018-01-07,534,652.6
7,2018-01-08,624,593.4
8,2018-01-09,652,606.0
9,2018-01-10,720,626.6


### Q6 - Day-of-week × temperature-bin breakdown (CTE)

Common-table expression bins the temperatures and tags weekdays first, then aggregates.

In [9]:
q6 = """
WITH binned AS (
    SELECT c.date,
           CASE strftime('%w', c.date)
               WHEN '0' THEN 'Sun' WHEN '1' THEN 'Mon' WHEN '2' THEN 'Tue'
               WHEN '3' THEN 'Wed' WHEN '4' THEN 'Thu' WHEN '5' THEN 'Fri'
               WHEN '6' THEN 'Sat'
           END AS dow,
           CASE
               WHEN w.tmax_f < 32 THEN '01_cold'
               WHEN w.tmax_f < 50 THEN '02_cool'
               WHEN w.tmax_f < 70 THEN '03_mild'
               WHEN w.tmax_f < 85 THEN '04_warm'
               ELSE '05_hot'
           END AS temp_bin,
           c.total_crimes
    FROM crime_daily c
    JOIN weather w ON c.date = w.date
)
SELECT dow, temp_bin,
       ROUND(AVG(total_crimes), 1) AS avg_crime,
       COUNT(*) AS n_days
FROM binned
GROUP BY dow, temp_bin
ORDER BY dow, temp_bin
"""
pd.read_sql(q6, conn)

,dow,temp_bin,avg_crime,n_days
0,Fri,01_cold,592.2,35
1,Fri,02_cool,670.3,95
2,Fri,03_mild,680.3,97
3,Fri,04_warm,744.8,91
4,Fri,05_hot,736.3,47
5,Mon,01_cold,565.2,31
6,Mon,02_cool,634.1,90
7,Mon,03_mild,652.7,98
8,Mon,04_warm,722.2,99
9,Mon,05_hot,741.6,48


In [10]:
conn.close()

Three tables populated (`weather`, `crime_daily`, `features`), all indexed on `date`. Six SQL queries return non-empty results and cover JOINs, CASE WHEN, window functions, and a CTE.